<a href="https://colab.research.google.com/github/pratik04032/pratik04032/blob/main/MobileNetV2_1e_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
sanikatiwarekar_deep_fake_detection_dfd_entire_original_dataset_path = kagglehub.dataset_download('sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset')

print('Data source import complete.')


  7%|▋         | 1.56G/22.5G [00:54<12:37, 29.7MB/s]

# **Dataset Paths (Kaggle Auto Mounted)**

In [ ]:
import kagglehub

# Download dataset
path = kagglehub.dataset_download(
    "sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset"
)

print("Dataset Path:", path)

# **DeepFake Detection — MobileNetV2 (Clean Structured Version)**

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# Performance optimization
tf.keras.mixed_precision.set_global_policy("mixed_float16")
tf.config.optimizer.set_jit(True)

In [ ]:
BASE_PATH = path

REAL_PATH = os.path.join(BASE_PATH, "DFD_original sequences")
FAKE_PATH = os.path.join(BASE_PATH, "DFD_manipulated_sequences", "DFD_manipulated_sequences")

assert os.path.exists(REAL_PATH)
assert os.path.exists(FAKE_PATH)

print("✅ Paths verified")

In [ ]:
IMG_SIZE = 128
FRAME_COUNT = 10
BATCH_SIZE = 32
CLASS_NAMES = ["Real", "Fake"]

# **Frame Extraction (Improved with Preprocess)**

In [ ]:
def extract_frames(video_path):

    cap = cv2.VideoCapture(video_path)
    frames = []

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total // FRAME_COUNT, 1)

    for i in range(FRAME_COUNT):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = preprocess_input(frame)
        frames.append(frame)

    cap.release()
    return np.array(frames)

# **Train-Test Split (Balanced)**

In [ ]:
real_videos = sorted(os.listdir(REAL_PATH))
fake_videos = sorted(os.listdir(FAKE_PATH))

fake_videos = fake_videos[:len(real_videos)]

real_train, real_test = train_test_split(real_videos, test_size=0.3, random_state=42)
fake_train, fake_test = train_test_split(fake_videos, test_size=0.3, random_state=42)

print("Train:", len(real_train), len(fake_train))
print("Test:", len(real_test), len(fake_test))

# **Load Videos**

In [ ]:
def load_videos(video_list, label, base_path):

    X, y, ids = [], [], []

    for vid in tqdm(video_list):
        frames = extract_frames(os.path.join(base_path, vid))
        if len(frames) != FRAME_COUNT:
            continue

        for f in frames:
            X.append(f)
            y.append(label)
            ids.append(vid)

    return np.array(X), np.array(y), np.array(ids)

# **Load Training Data**

In [ ]:
X_real, y_real, _ = load_videos(real_train, 0, REAL_PATH)
X_fake, y_fake, _ = load_videos(fake_train, 1, FAKE_PATH)

X_train = np.concatenate([X_real, X_fake])
y_train = np.concatenate([y_real, y_fake])

# **Data Augmentation**

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1)
])

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(5000)
train_ds = train_ds.map(lambda x,y:(augment(x),y), num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# **Model Builder (3 Versions)**

In [ ]:
def build_model(drop=0.3, l2v=0.01):

    base = MobileNetV2(weights="imagenet",
                       include_top=False,
                       input_shape=(IMG_SIZE,IMG_SIZE,3))

    base.trainable = True

    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dropout(drop),
        Dense(256, activation="relu", kernel_regularizer=l2(l2v)),
        Dropout(drop),
        Dense(1, activation="sigmoid", dtype=tf.float32)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

# **Train Models**

In [ ]:
models = {
    "V1": build_model(0.3,0.01),
    "V2": build_model(0.2,0.0),
    "V3": build_model(0.5,0.02)
}

early = EarlyStopping(patience=5, restore_best_weights=True)

histories = {}

for name,m in models.items():
    print(name)
    h = m.fit(train_ds, epochs=20, callbacks=[early])
    histories[name]=m

# **Load Test Data**

In [ ]:
Xr, yr, idr = load_videos(real_test,0,REAL_PATH)
Xf, yf, idf = load_videos(fake_test,1,FAKE_PATH)

X_test = np.concatenate([Xr,Xf])
y_test = np.concatenate([yr,yf])
vid_ids = np.concatenate([idr,idf])

# **Evaluation (Frame + Video Level)**

In [ ]:
results=[]

for name,model in histories.items():

    probs = model.predict(X_test).flatten()
    preds = (probs>0.5).astype(int)

    frame_acc = accuracy_score(y_test,preds)

    video_preds=defaultdict(list)
    labels={}

    for i,v in enumerate(vid_ids):
        video_preds[v].append(probs[i])
        labels[v]=y_test[i]

    vtrue=[]
    vpred=[]
    vprobs=[]

    for v,p in video_preds.items():
        avg=np.mean(p)
        vpred.append(int(avg>0.5))
        vtrue.append(labels[v])
        vprobs.append(avg)

    video_acc=accuracy_score(vtrue,vpred)

    fpr,tpr,_=roc_curve(y_test,probs)
    roc_auc=auc(fpr,tpr)

    results.append([name,frame_acc,video_acc,roc_auc])

df=pd.DataFrame(results,columns=["Model","FrameAcc","VideoAcc","ROC"])
df.sort_values("VideoAcc",ascending=False)

# **Comparison graphs (Frame vs Video, Loss, ROC AUC)**

In [ ]:
results_df = pd.DataFrame(results, columns=[
    "Model",
    "Frame Accuracy",
    "Video Accuracy",
    "ROC AUC"
])

plt.figure(figsize=(8,5))
x = np.arange(len(results_df["Model"]))
bar_width = 0.35

plt.bar(x - bar_width/2, results_df["Frame Accuracy"], width=bar_width, label="Frame Accuracy")
plt.bar(x + bar_width/2, results_df["Video Accuracy"], width=bar_width, label="Video Accuracy")

plt.xticks(x, results_df["Model"])
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# **Save Best Model**

In [ ]:
best_model = histories["V1"]   # choose best
best_model.save("mobilenet_deepfake.h5")

In [ ]:
pip install yt-dlp

In [ ]:
import os
import requests
import yt_dlp

# **External URL Prediction Support**

In [ ]:


def download_video(url, save_path="temp_video.mp4"):

    ydl_opts = {
        'format': 'mp4',
        'outtmpl': save_path,
        'quiet': True
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    if not os.path.exists(save_path) or os.path.getsize(save_path) < 100000:
        raise Exception("Video download failed.")

    return save_path

In [ ]:
def predict_video_from_path(video_path, model):

    frames = extract_frames(video_path)

    if len(frames) == 0:
        return {"Prediction": "Invalid Video File", "Confidence": 0}

    probs = model.predict(frames).flatten()
    avg_prob = np.mean(probs)

    label = "Fake" if avg_prob > 0.5 else "Real"
    confidence = float(avg_prob if avg_prob > 0.5 else 1 - avg_prob)

    return {
        "Prediction": label,
        "Confidence": round(confidence * 100, 2)
    }

In [ ]:
def predict_from_url(url, model):

    print("Downloading...")
    video_path = download_video(url)

    result = predict_video_from_path(video_path, model)

    os.remove(video_path)

    return result

# **Video_path**

In [ ]:
best_model.save("/kaggle/working/mobilenet_deepfake.h5")
model = tf.keras.models.load_model("mobilenet_deepfake.h5")

url = input("Provided Url here:")

result = predict_from_url(url, model)

print(result)

In [ ]:
video_path =  input("Provided you video path:")

result = predict_video_from_path(video_path, model)

print(result)